# 🧬 缝合 + 愈合实战：18B Frankenmerge 与 QLoRA 愈合

> **来源说明**：jackrong 仓库**没有发布** `frankenmerge.py` / `heal-frankenmerge.py` 的完整脚本，但官方公布了：
> - 完整技术流程文档 **`reference/MERGE_PROCESS.md`**（HF `Jackrong/Qwopus-GLM-18B-Merged-GGUF` 仓库，作者 Kyle Hessling）；
> - 愈合后模型权重 `Jackrong/Qwopus-GLM-18B-Healed`（BF16）+ GGUF 量化版；
> - 后续 Fusion 项目的**完整合并脚本 `reference/merge_layerweighted.py`**（本 notebook 的工程骨架来源）。
>
> 本 notebook = 官方文档主线 + 官方脚本工程骨架的**对齐实现**（合成 checkpoint 已在 CPU 验证，见第 4 节输出）。

| 项目 | 内容 |
|---|---|
| 🧬 合并方式 | Passthrough Frankenmerge：两个 9B 的 32 层顺序堆叠成 64 层 18B |
| 🩹 愈合方式 | 1000 步 QLoRA（r=64，346M 可训练参数 = 3.62%） |
| 📊 官方成绩 | 缝合 39/44 → 愈合 40/44；超过 Qwen3.6-35B MoE（38/44），显存只要一半（9.2GB vs 22GB） |
| 💻 资源需求 | 合并：不用 GPU（40GB RAM，15 分钟）；愈合：RTX 5090，~13GB 显存，~14 小时 |

**目录**
1. 为什么缝合：能力-显存权衡
2. 原理：passthrough 层堆叠 + 层边界表征不连续
3. mergekit 为什么失败
4. 缝合执行（官方工程骨架的完整实现，CPU 可跑）
5. GGUF 转换与量化
6. 愈合原理与官方配置
7. 愈合执行（QLoRA 训练）
8. 愈合后评估
9. 官方复盘
10. 复现 7 步总览

## 1️⃣ 为什么缝合：能力-显存权衡

9B 微调模型跑得动但能力有天花板；27B 能力强但 Q4 都要 16GB+ 显存。
**把两个互补的 9B 微调模型"缝"成一个 18B**，正好填上 12-16GB 消费卡的中间地带：

| 模型 | 44 项测试 | Q4_K_M 体积 | 吞吐 | 单题耗时 |
|---|---|---|---|---|
| Qwopus3.5-9B（源 A） | **41/44 (93.2%)** | 5.3 GB | 126.0 tok/s | 55s |
| **Qwopus-GLM-18B 缝合版** | 39/44 (88.6%) | 9.2 GB | 66.6 tok/s | 127s |
| Qwen 3.6-35B MoE | 38/44 (86.4%) | 22 GB | 174.2 tok/s | 442s |

**官方关键结论**：原始 18B 缝合版就超过 35B MoE，**显存不到一半、耗时仅 29%**。
两个源模型的选择是刻意的互补（官方文档）：
- **Qwopus3.5-9B-v3.5**（A）：Opus 风格推理蒸馏——agentic、工具调用、结构化代码；
- **Qwen3.5-9B-GLM5.1-Distill-v1**（B）：GLM 风格蒸馏——层次化问题分解、指令遵循。

## 2️⃣ 原理：passthrough 层堆叠 + 层边界表征不连续

**Passthrough** 是最简单的缝合：不做任何权重插值，把两个模型的 transformer 层顺序拼接：

```
新模型（64 层）:
  第  0-31 层  ← A（Qwopus3.5-9B-v3.5）的全部 32 层
  第 32-63 层  ← B（Qwen3.5-9B-GLM5.1-Distill-v1）的全部 32 层
  Embedding / LM head / MTP / 视觉编码器 ← 全部取 A
```

为什么不用 SLERP/TIES/DARE？官方解释：这些方法在**同层位置**插值权重，产出还是 9B；
只有 passthrough 能**加深网络**（参数翻倍）。两个源共享 Qwen3.5-9B 基座，
权重空间大致对齐，直接拼接是合理的选择。

**但是——缝合处会生病**（官方 Figure 2 机制）：

```
       第 31 层（A 权重）        第 32 层（B 权重）
  ────────────────────┤ 表征不连续 ├────────────────────
   A 的输出分布 ≠ B 期望的输入分布 → 激活失真
   症状：代码围栏丢失 / 括号丢失 / 函数名错乱 / 幻觉语法
```

- 症状集中在**结构化输出**（需要跨层紧密的 token 级协作）；推理/agentic 能力几乎不受影响——高层语义表征对层堆叠更鲁棒；
- 这就是"愈合"要治的病：让低秩适配器在边界层学会翻译两种表征。

## 3️⃣ mergekit 为什么失败

官方文档记录了踩坑过程：mergekit 无法处理 Qwen3.5 架构，两个原因：

1. **多模块架构**：Qwen3.5 是 `Qwen3_5ForConditionalGeneration`，含 4 个模块——
   `model.language_model.layers`（32 层）+ `default`（16 个散张量）+ `mtp.layers`（MTP 层）+ `model.visual.blocks`（27 个视觉编码器块）；
2. **混合注意力**：linear_attn（SSM 式）与 self_attn（标准）**每 4 层交替**，mergekit 的层重编号逻辑在两种层类型间搞混张量映射，报错：
   ```
   RuntimeError: Tensor model.language_model.layers.3.linear_attn.out_proj.weight
   required but not present in model
   ```

**结论（官方）**：合并脚本必须**架构无关**——只按张量名正则处理，不关心层结构。
官方后续 Fusion 项目的 `merge_layerweighted.py` 沿用了同一思路（`safe_open` 流式读 + `ShardWriter` 分片写）。

## 4️⃣ 缝合执行（官方工程骨架的完整实现）

**本 cell 的代码 = 官方两处资源的对齐合并**，CPU 就能跑：
- 层重编号规则：MERGE_PROCESS.md 的 `renumber_layer`（前缀正则，逐字保留）；
- 工程骨架：`reference/merge_layerweighted.py` 的 `safe_open` 惰性句柄缓存 + `ShardWriter`（5GB 分片、重命名、index.json）。

**官方输出日志**（真实 18B 缝合）：A 775 张量保留 + B 424 张量重编号 = 1199 张量；
7 分片 33.15GB；~40GB RAM 峰值；~15 分钟（几乎全是 I/O）；**不用 GPU**。
下方先跑合成 checkpoint 自测（Qwen3.5 多模块 + hybrid attention 结构），真实合并见下一个 cell。

In [1]:
import json
import re
import shutil
from pathlib import Path

import torch
from safetensors import safe_open
from safetensors.torch import save_file

# ============================================================
# 官方 MERGE_PROCESS.md 原样代码片段：按前缀重编号 B 的层张量
# ============================================================
def renumber_layer(key: str, offset: int, prefix: str):
    pattern = rf'^({re.escape(prefix)}\.)(\d+)(\..*)'
    m = re.match(pattern, key)
    if m:
        new_idx = int(m.group(2)) + offset
        return f"{m.group(1)}{new_idx}{m.group(3)}"
    return None

LAYER_PREFIX = "model.language_model.layers"   # Qwen3.5 系
SHARD_BYTES = 5_000_000_000                    # 官方：5 GB per shard

# ============================================================
# 工程骨架：参考官方 merge_layerweighted.py（safe_open 惰性句柄 + ShardWriter）
# ============================================================
def load_index(p):
    return {k: Path(p) / v for k, v in
            json.load(open(f"{p}/model.safetensors.index.json"))["weight_map"].items()}

def _iter_tensors(model_dir: Path):
    """按 index 顺序产出 (tensor_name, file_path)；无 index 则扫描单文件。"""
    idx = model_dir / "model.safetensors.index.json"
    if idx.exists():
        for name, fpath in load_index(model_dir).items():
            yield name, fpath
    else:
        for f in sorted(model_dir.glob("*.safetensors")):
            with safe_open(f, framework="pt", device="cpu") as h:
                for name in h.keys():
                    yield name, f

class ShardWriter:
    """5GB 分片写出 + 重命名 + index.json（官方 merge_layerweighted.py 同款）。"""
    def __init__(self, out_dir: Path):
        self.out = out_dir; self.out.mkdir(parents=True, exist_ok=True)
        self.buf, self.cur, self.i, self.wm = {}, 0, 0, {}
    def _flush(self):
        if not self.buf: return
        self.i += 1
        nm = f"model-{self.i:05d}.safetensors"
        save_file(self.buf, self.out / nm, metadata={"format": "pt"})
        for k in self.buf: self.wm[k] = nm
        self.buf, self.cur = {}, 0
    def add(self, name, tensor):
        self.buf[name] = tensor.contiguous()
        self.cur += tensor.numel() * tensor.element_size()
        if self.cur >= SHARD_BYTES: self._flush()
    def finalize(self):
        self._flush()
        n = self.i
        for j in range(1, n + 1):
            old, new = f"model-{j:05d}.safetensors", f"model-{j:05d}-of-{n:05d}.safetensors"
            (self.out / old).rename(self.out / new)
            for k, v in list(self.wm.items()):
                if v == old: self.wm[k] = new
        json.dump({"metadata": {"total_size": 0}, "weight_map": self.wm},
                  open(self.out / "model.safetensors.index.json", "w"), indent=2)

# ============================================================
# 缝合主流程（官方规则：A 全部保留；B 仅层张量重编号；config 层数+layer_types 翻倍）
# ============================================================
def passthrough_merge(model_a: str, model_b: str, out_dir: str,
                      layer_prefix: str = LAYER_PREFIX) -> None:
    a_dir, b_dir = Path(model_a), Path(model_b)
    out = Path(out_dir)

    config = json.loads((a_dir / "config.json").read_text())
    a_layers = config["num_hidden_layers"]
    b_config = json.loads((b_dir / "config.json").read_text())
    b_layers = b_config["num_hidden_layers"]

    a_names = list(_iter_tensors(a_dir))
    b_names = list(_iter_tensors(b_dir))
    assert len(a_names) == len(b_names), f"两边张量数不一致：{len(a_names)} vs {len(b_names)}"

    # safe_open 惰性句柄缓存（官方 merge_layerweighted.py 同款：避免反复打开分片）
    handles = {}
    def get(path, name):
        if path not in handles:
            handles[path] = safe_open(path, framework="pt", device="cpu")
        return handles[path].get_tensor(name)

    writer = ShardWriter(out)
    n_renumbered = 0
    for name, f in a_names:
        writer.add(name, get(f, name))                       # A：全部保留
    for name, f in b_names:
        new_name = renumber_layer(name, a_layers, layer_prefix)
        if new_name is None:
            continue                                         # B 的 embed/LM head 等被 A 覆盖
        writer.add(new_name, get(f, name))
        n_renumbered += 1

    config["num_hidden_layers"] = a_layers + b_layers        # 官方输出日志：32 -> 64
    if "layer_types" in config:
        if len(config["layer_types"]) == a_layers:           # layer_types 翻倍（官方日志）
            config["layer_types"] = config["layer_types"] + b_config.get(
                "layer_types", config["layer_types"])
        else:
            print(f"⚠️ layer_types 长度 {len(config['layer_types'])} ≠ {a_layers}，跳过")
    json.dump(config, open(out / "config.json", "w"), indent=2, ensure_ascii=False)
    writer.finalize()

    # tokenizer / 模板等文件取 A
    for fn in ["tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt",
               "special_tokens_map.json", "generation_config.json",
               "preprocessor_config.json", "chat_template.jinja"]:
        if (a_dir / fn).exists():
            shutil.copy2(a_dir / fn, out / fn)

    print(f"✅ 缝合完成：{a_layers} 层 (A) + {b_layers} 层 (B) = {a_layers + b_layers} 层 → {out}")
    print(f"   A 保留 {len(a_names)} 张量，B 重编号 {n_renumbered} 张量，"
          f"输出 {writer.i} 分片 + index.json")

print("✅ 缝合实现就绪：renumber_layer（官方片段）+ safe_open + ShardWriter（官方骨架）")

✅ 缝合实现就绪：renumber_layer（官方片段）+ safe_open + ShardWriter（官方骨架）


In [2]:
# ============================================================
# 合成 checkpoint 自测（本机 CPU）：Qwen3.5 多模块 + hybrid attention mock
# 验证六件事：前缀重编号 / 多模块张量取 A / hybrid 层随层平移 / layer_types 翻倍
#            / 分片与 index.json / B 的非层张量被正确丢弃
# ============================================================
import tempfile

def make_qwen_mock(d, n_layers, tag, dim=8):
    d = Path(d); d.mkdir(parents=True, exist_ok=True)
    cfg = {"architectures": ["Qwen3_5ForConditionalGeneration"],
           "num_hidden_layers": n_layers, "hidden_size": dim, "vocab_size": 100,
           "layer_types": ["full_attention"] * n_layers}
    json.dump(cfg, open(d / "config.json", "w"))
    tensors = {}
    v = 1 if tag == "A" else 2
    tensors["model.embed_tokens.weight"] = torch.ones(dim, 100) * v
    tensors["lm_head.weight"] = torch.ones(100, dim) * v
    tensors["model.visual.blocks.0.weight"] = torch.ones(dim) * v     # 视觉编码器（无层号）
    tensors["model.mtp_layers.0.weight"] = torch.ones(dim) * v        # MTP 层（.layers. 但不是 language_model）
    for i in range(n_layers):
        for p in ["self_attn.q_proj.weight", "mlp.gate_proj.weight"]:
            tensors[f"model.language_model.layers.{i}.{p}"] = torch.full((dim, dim), i)
        if i % 4 == 0:                                                # hybrid：每 4 层 linear_attn
            tensors[f"model.language_model.layers.{i}.linear_attn.in_proj_a.weight"] = \
                torch.full((dim, dim), 100 + i)
    save_file(tensors, d / "model.safetensors")

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    make_qwen_mock(tmp / "mockA", 2, "A")
    make_qwen_mock(tmp / "mockB", 2, "B")
    passthrough_merge(str(tmp / "mockA"), str(tmp / "mockB"), str(tmp / "mockMerged"))

    from safetensors.torch import load_file
    m = load_file(tmp / "mockMerged" / "model-00001-of-00001.safetensors")
    cfg = json.load(open(tmp / "mockMerged" / "config.json"))

    assert (m["model.embed_tokens.weight"] == 1).all(), "embed 应取 A"
    assert (m["lm_head.weight"] == 1).all(), "LM head 应取 A"
    assert (m["model.visual.blocks.0.weight"] == 1).all(), "视觉编码器应取 A"
    assert (m["model.mtp_layers.0.weight"] == 1).all(), "MTP 应取 A（未被当 transformer 层重编号）"
    for i in [0, 1]:
        assert (m[f"model.language_model.layers.{i}.mlp.gate_proj.weight"] == i).all()
    for i in [2, 3]:
        assert (m[f"model.language_model.layers.{i}.mlp.gate_proj.weight"] == i - 2).all()
    assert (m["model.language_model.layers.0.linear_attn.in_proj_a.weight"] == 100).all()
    assert (m["model.language_model.layers.2.linear_attn.in_proj_a.weight"] == 100).all()
    assert cfg["num_hidden_layers"] == 4
    assert cfg["layer_types"] == ["full_attention"] * 4
    assert (tmp / "mockMerged" / "model.safetensors.index.json").exists()

print("✅ 六项断言全部通过：前缀重编号 / 多模块取 A / hybrid 随层平移 / layer_types 翻倍 / 分片 / index")

✅ 缝合完成：2 层 (A) + 2 层 (B) = 4 层 → /var/folders/04/mzk2qmr50hz5_dgz8tqbb6v80000gn/T/tmprxs1svfx/mockMerged
   A 保留 9 张量，B 重编号 5 张量，输出 1 分片 + index.json
✅ 六项断言全部通过：前缀重编号 / 多模块取 A / hybrid 随层平移 / layer_types 翻倍 / 分片 / index


In [ ]:
# ============================================================
# 真实缝合（官方资源用量：40GB RAM 峰值、33GB 输出、~15 分钟、不用 GPU）
# 本 cell 未在本机执行（需下载两个 9B 模型，~40GB），Colab/本地大内存机器直接运行。
# ============================================================
# !pip install torch safetensors
# passthrough_merge(
#     "/path/to/Jackrong/Qwopus3.5-9B-v3.5",          # 模型 A（含全部 tokenizer/config 文件）
#     "/path/to/Jackrong/Qwen3.5-9B-GLM5.1-Distill-v1", # 模型 B
#     "/path/to/Qwopus-GLM-18B-merged",                 # 官方输出路径样式
# )
# # 预期输出（官方日志）：
# #   ✅ 缝合完成：32 层 (A) + 32 层 (B) = 64 层
# #      A 保留 775 张量，B 重编号 424 张量，输出 7 分片 + index.json

## 5️⃣ GGUF 转换与量化（官方命令）

官方全流程：BF16 GGUF 30GB（851 张量，40 秒）→ Q4_K_M 9.2GB（44 秒，压缩比 3.2×）。
注意官方 serving 用了**修补版 Jinja 模板**（修标准 Qwen3.5 模板处理字符串参数时的 `|items` bug）。

In [ ]:
# ---- Step 1: safetensors → GGUF (BF16) ----
# !python llama.cpp/convert_hf_to_gguf.py ~/models/Qwopus-GLM-18B-merged \
#     --outfile ~/models/Qwopus-GLM-18B-merged-f16.gguf --outtype bf16

# ---- Step 2: 量化 Q4_K_M（官方：9.2 GB，3.2× 压缩，44 秒）----
# !llama.cpp/llama-quantize ~/models/Qwopus-GLM-18B-merged-f16.gguf \
#     ~/models/Qwopus-GLM-18B-merged-Q4_K_M.gguf Q4_K_M

# ---- Step 3: 本地服务（官方命令原样）----
# !llama.cpp/llama-server -m ~/models/Qwopus-GLM-18B-merged-Q4_K_M.gguf \
#     --alias "Qwopus-GLM-18B" --chat-template-file ~/.hermes/qwen35-fixed.jinja \
#     --host 127.0.0.1 --port 8001 --ctx-size 65536 --flash-attn on --n-gpu-layers 99 \
#     --cache-type-k q8_0 --cache-type-v q8_0 --batch-size 8192 --ubatch-size 4096 \
#     --parallel 1 --mlock --threads 20 --threads-batch 20

## 6️⃣ 愈合原理与官方配置

**原理**（官方 Figure 2）：第 32 层边界处 A 的表征与 B 期望的输入分布错位。
愈合 = 在边界层挂低秩适配器 `ΔW = A·B`，让模型自己学会"翻译"两种表征。
官方证据：**前 100 步 loss 从 1.02 骤降到 0.72**——如果是普通质量噪声不会降这么快，
说明愈合在治本（边界是离散可学习的误差源）。

**官方配置表（MERGE_PROCESS.md 8.2 节，逐项对应下方代码）**：

| 参数 | 值 |
|---|---|
| 方法 | QLoRA（4-bit NF4 + 双量化） |
| LoRA rank / α | 64 / 32 |
| 可训练参数 | 346M / 9.5B quantized（3.62%） |
| 学习率 | 2e-5（cosine） |
| Warmup | 50 步 |
| Batch | 8（2/卡 × 4 梯度累积） |
| 步数 / 序列长度 | 1000 / 4096 |
| 显存 | ~13 GB allocated / ~17.5 GB reserved |
| 时长 | ~14 小时（RTX 5090） |

**TARGET_MODULES**（官方原样——注意覆盖 linear_attn 与 self_attn 两种层类型）：

```python
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj_a", "in_proj_b", "in_proj_z", "in_proj_qkv", "out_proj",
    "gate_proj", "up_proj", "down_proj",
]
```

**愈合数据**（官方 8.3，~1,383 条 70/15/15）：`Qwen3.5-reasoning-700x`（70%）+
`Competitive-Programming-python-blend`（15%，代码样本针对格式退化）+ `MultiReason-ChatAlpaca`（15%）。

## 7️⃣ 愈合执行（QLoRA 训练）

下方为官方配置的完整训练代码（Unsloth）。官方 CLI 用法：

```bash
python3 heal-frankenmerge.py --dry-run                 # 先验证一切能加载
python3 heal-frankenmerge.py --max-steps 1000 --num-samples 5000   # 完整愈合 ~14h
```

数据归一化复用了上篇（`cook_qwopus_sft.ipynb`）验证过的同一套 schema 适配器。
官方 loss 曲线（愈合有效性的直接证据）：

| Step | Loss | 备注 |
|---|---|---|
| 10 | 1.0175 | 初始（边界混乱） |
| 50 | 0.8758 | 早期快速下降 |
| 100 | 0.7215 | 边界开始愈合 |
| 500 | 0.6154 | Checkpoint 2（loss 稳定） |
| 1000 | 0.6396 | 最终（**-39%**） |

In [ ]:
# ============================================================
# 官方配置的愈合训练（GPU；在 Colab/本地大显存机器运行）
# 依赖：pip install unsloth trl datasets（上篇第 3 节同款安装）
# ============================================================
import re
from dataclasses import dataclass

# ---- 官方 TARGET_MODULES（8.2 节原样）----
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj_a", "in_proj_b", "in_proj_z", "in_proj_qkv", "out_proj",
    "gate_proj", "up_proj", "down_proj",
]

@dataclass
class HealConfig:
    merged_model: str = "./Qwopus-GLM-18B-merged"   # 第 4 节缝合输出
    max_seq_length: int = 4096                      # 官方 8.2
    lora_r: int = 64
    lora_alpha: int = 32
    lr: float = 2e-5                                # cosine
    warmup_steps: int = 50
    per_device_batch_size: int = 2                  # 2 × 4 = 有效 batch 8
    gradient_accumulation_steps: int = 4
    max_steps: int = 1000                           # CLI 可覆盖（--max-steps 5000 也支持）
    total_samples: int = 1383                       # 官方 8.3；CLI --num-samples 可加大
    seed: int = 3407

In [ ]:
# ---- 愈合数据（官方 8.3 配比；schema 适配器与上篇第 6 节同一套，已验证）----
def _strip(x):
    return (x or "").strip()

def normalize_assistant(text: str) -> str:
    text = _strip(text)
    if not text:
        return "<think></think>\n"
    m = re.search(r"<think>.*?</think>", text, flags=re.DOTALL)
    if m:
        block = m.group(0).strip()
        rest = text[m.end():].lstrip()
        return f"{block}\n{rest}".rstrip() if rest else f"{block}\n"
    return f"<think></think>\n{text}".rstrip()

def build_heal_dataset(cfg):
    """700x 70% + Competitive-Programming 15% + MultiReason 15%（官方 8.3）。"""
    from datasets import Dataset, load_dataset

    def sample(name, n):
        ds = load_dataset(name, split="train")
        n = min(n, len(ds))
        return ds.shuffle(seed=cfg.seed).select(range(n))

    n1, n2 = int(cfg.total_samples * 0.70), int(cfg.total_samples * 0.15)
    n3 = cfg.total_samples - n1 - n2
    ds1 = sample("Jackrong/Qwen3.5-reasoning-700x", n1)                    # conversation from/value
    ds3 = sample("Jackrong/MultiReason-ChatAlpaca", n3)                    # conversation from/value
    ds2 = sample("Jackrong/Competitive-Programming-python-blend", n2)      # messages

    convos = []
    for conv in list(ds1["conversation"]) + list(ds3["conversation"]):
        cleaned = []
        for m in conv:
            frm = (m.get("from") or "").strip()
            if frm == "human":
                cleaned.append({"role": "user", "content": _strip(m.get("value", ""))})
            elif frm == "gpt":
                cleaned.append({"role": "assistant",
                                "content": normalize_assistant(m.get("value", ""))})
        if len(cleaned) >= 2 and cleaned[-1]["role"] == "assistant":
            convos.append(cleaned)
    for msgs in ds2["messages"]:
        convo = [m for m in msgs if isinstance(m, dict) and m.get("role") != "system"]
        cleaned = []
        for m in convo:
            content = m.get("content", "")
            if m.get("role") == "assistant":
                content = normalize_assistant(content)
            if m.get("role") in ("user", "assistant") and _strip(content):
                cleaned.append({"role": m["role"], "content": _strip(content)})
        if len(cleaned) >= 2 and cleaned[-1]["role"] == "assistant":
            convos.append(cleaned)

    from datasets import Dataset
    dataset = Dataset.from_list({"conversations": convos}).shuffle(seed=cfg.seed)

    def to_text(examples):
        return {"text": [tokenizer.apply_chat_template(c, tokenize=False,
                                                       add_generation_prompt=False)
                         for c in examples["conversations"]]}
    dataset = dataset.map(to_text, batched=True)
    dataset = dataset.filter(lambda b: [len(tok) <= cfg.max_seq_length for tok in
        tokenizer(b["text"], truncation=False, padding=False,
                  add_special_tokens=False)["input_ids"]], batched=True)
    print(f"愈合数据就绪：{len(dataset)} 条（{n1}+{n2}+{n3} 采样，过滤后）")
    return dataset

In [ ]:
# ---- 训练（官方配置逐项对应）----
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

cfg = HealConfig()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.merged_model,
    max_seq_length = cfg.max_seq_length,
    load_in_4bit = True,                     # 官方：QLoRA 4-bit NF4 + 双量化
)
model = FastLanguageModel.get_peft_model(
    model,
    r = cfg.lora_r,                          # 官方：64
    target_modules = TARGET_MODULES,         # 官方原样（linear_attn + self_attn 全覆盖）
    lora_alpha = cfg.lora_alpha,             # 官方：32
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = cfg.seed,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")

dataset = build_heal_dataset(cfg)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = cfg.per_device_batch_size,
        gradient_accumulation_steps = cfg.gradient_accumulation_steps,
        warmup_steps = cfg.warmup_steps,
        max_steps = cfg.max_steps,
        learning_rate = cfg.lr,
        lr_scheduler_type = "cosine",        # 官方：cosine schedule
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        seed = cfg.seed,
        save_steps = 250,                    # 官方：checkpoints every 250 steps
        save_total_limit = 2,
        report_to = "none",
        output_dir = "./heal_out",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>",
)

# --dry-run 等价操作：先注释掉下一行，验证数据/模型构建无误后再放行
trainer.train()

# ---- 收尾：保存 LoRA + 合并 16bit 推送（可选）----
model.save_pretrained("./heal_out/heal_lora")
tokenizer.save_pretrained("./heal_out/heal_lora")
# model.push_to_hub_merged("你的用户名/Qwopus-GLM-18B-healed",
#                          tokenizer, save_method="merged_16bit", token=HF_TOKEN)

## 8️⃣ 愈合后评估（官方结果）

**44 项基准**（9 类：基础/推理/工具调用/agentic/结构化/上下文/多语/编程/性能）：

| 类别 | 缝合版 | 愈合版 | 变化 |
|---|---|---|---|
| 编程 | 11/15 | **12/15** | +1（找回 `longest_substring`，滑动窗口 8/8 用例通过） |
| 其余 8 类 | 28/29 | 28/29 | 持平（推理/工具/agentic 本就满血） |
| **总计** | 39/44 (88.6%) | **40/44 (90.9%)** | +1 |

**前端压力测试**（愈合的价值在长结构化输出上才真正体现——6 个复杂 HTML/CSS/JS 任务，63 项可执行检查）：

| 任务 | 检查 | 得分 | 输出规模 |
|---|---|---|---|
| Weather Dashboard | 9 | 9/9 | 14.5K 字符 |
| E-Commerce Product Page | 12 | 12/12 | 16.7K 字符 |
| Animated SaaS Landing | 13 | 13/13 | 24.1K 字符 |
| Analytics Dashboard | 13 | 13/13 | 22.3K 字符 |
| Multi-Step Registration | 12 | 12/12 | 23.3K 字符 |
| Snake Game | 12 | 11/12 | 11.2K 字符 |
| **总计** | **63** | **62/63 (98.4%)** | 括号/花括号完全配平，零乱码 |

## 9️⃣ 官方复盘（Lessons Learned）

1. **QLoRA 只改 2% 参数就能治缝合病**：边界是离散可学习的误差源，1,383 条定向数据足够；
2. **短基准上 9B 源模型反而最高（41/44）**：单轮短 prompt 下精调小模型能赢未精炼的大缝合；
   18B 的优势在长、复杂、结构化输出（前端压力测试）；
3. **愈合后仍有 3 个编程测试失败**：函数命名与括号问题残留——1,383 条样本不够修完所有格式边角。

**官方"如果重来"**：
1. **更多代码数据**——750 条竞赛编程样本不够，代码专向数据集能补完剩余差距；
2. **面向长输出的评测**——44 项套件用短 prompt，低估了 18B 的优势场景；
3. **交错堆叠**（A[0],B[0],A[1],B[1]...）替代顺序堆叠——把合并边界摊到每一层，或可减少愈合需求；
4. **多卡全参愈合**代替 QLoRA——100% 参数训练容量最大，但 QLoRA 效果已足够好。

## 🔟 复现 7 步总览（官方第 10 节）

```bash
# 0. 环境
pip install torch safetensors huggingface_hub datasets    # 合并
pip install unsloth bitsandbytes trl peft accelerate      # 愈合

# 1. 缝合（本 notebook 第 4 节；~15 分钟，不用 GPU）
passthrough_merge("A目录", "B目录", "~/models/Qwopus-GLM-18B-merged")

# 2-3. GGUF 转换 + Q4_K_M 量化（第 5 节；~2 分钟）
# 4. 基准：llama-server + tests/test_qwopus_v35.py（官方 44 项套件）

# 5. 愈合（第 7 节；~14h on RTX 5090）
python3 heal-frankenmerge.py --dry-run
python3 heal-frankenmerge.py --max-steps 1000 --num-samples 5000

# 6-7. 愈合版同样转 GGUF + 跑基准对比
```

**官方 File Index**（原项目文件，供寻找上游时对照）：`frankenmerge.py` / `heal-frankenmerge.py` /
`tests/test_qwopus_v35.py` / `merge-config.yaml` / `qwen35-fixed.jinja`——其中前两个官方未公开，
本 notebook 第 4/7 节即其对齐实现。

## 1️⃣1️⃣ 附录

### A. 官方 merge_layerweighted.py（本 notebook 工程骨架来源）

`reference/merge_layerweighted.py` 是官方**后续 Fusion 项目公开的完整合并脚本**（78 行），
与本篇 passthrough 不同，它做的是 **layer-weighted delta merge**（基于实测权重几何）：

```
W_new = V2 + α(L) × (Coder − V2)
α(L) 按深度线性斜坡：浅层低（保护推理能力）→ 深层高（注入编码能力）
embed / lm_head / norm / mtp / vision 冻结，全部复制自 V2
```

值得借鉴的工程件（本 notebook 已沿用）：`safe_open` 惰性句柄、`ShardWriter` 5GB 分片 +
index.json、`load_index`、COPY 正则。想做**三模型加权合并**时直接读该文件。

### B. 故障排查

| 症状 | 可能原因 | 处理 |
|---|---|---|
| 缝合后无法加载 | 两边张量数不一致（架构不同） | 脚本已断言；换同基座模型 |
| mergekit 报 `required but not present` | hybrid attention 层重编号错乱 | 用本 notebook 的架构无关脚本 |
| 愈合 loss 不降 | 数据格式/长度过滤问题 | 对照第 7 节 `build_heal_dataset` 产物 |
| 输出仍有格式乱码 | 愈合样本不够（官方：1383 条修不完所有边角） | `--num-samples 5000` + 更多代码数据 |

### C. 参考

1. `reference/MERGE_PROCESS.md` —— 官方合并+愈合完整技术流程（本教程主线）
2. `reference/merge_layerweighted.py` —— 官方完整合并脚本（工程骨架）
3. `reference/Qwopus-GLM-18B-Technical-Report.pdf` —— 官方技术报告
4. HF：`Jackrong/Qwopus-GLM-18B-Healed`（愈合权重）、`Jackrong/Qwopus-GLM-18B-Merged-GGUF`（量化）
5. 缝合数据三件套详情见 `food/post-training.md`（700x / Competitive-Programming / MultiReason）

---

🎉 **三篇结束**。你已掌握：上篇省钱 SFT 五件套（`cook_qwopus_sft.ipynb`）→ 中篇 R1-Zero GRPO（`cook_r1_grpo.ipynb`）→ 本篇缝合+愈合。三者组合即 jackrong 社区"单卡后训练"的完整技能树。